# School & Education Challenges — Topic Clustering (full-batch, max accuracy)

Clusters the **real** `Challenges` data (one document per pipe-delimited challenge) with the
`graphweave` library, then produces **interactive, shareable HTML maps** for a non-technical audience,
with the **District** dimension overlaid.

**Pipeline:** load → explode challenges → embed (cached) → full-batch fit (fine-grained) →
LLM labels → LLM merge into parent topics → results tables → `datamapplot` atlas + Plotly map.

### Run this on Kaggle
1. **Settings → Accelerator → GPU** (T4 ×2 or P100).
2. **Settings → Internet → ON** (needed for `pip install` and OpenRouter).
3. Upload the CSV(s) as a Kaggle **Dataset** (it mounts under `/kaggle/input/...`) and point
   `CSV_PATHS` at them (a list) in the config cell.
4. Add your **OpenRouter key** via **Add-ons → Secrets** as `OPENROUTER_API_KEY`
   (the notebook reads it automatically; without it, it falls back to keyword labels + single level).

> **Why full-batch, not the batch/coreset path?** ~79k documents sit far below the library's
> `max_inmemory_docs=300,000` regime switch and the embeddings are tiny (~230 MB). The
> cumulative/coreset machinery *approximates* the corpus to bound memory on unbounded streams — it
> trades accuracy for memory. With the whole corpus in hand and accuracy as the goal, a single
> `GraphWeave.fit()` over the complete consensus-Leiden graph is strictly the better choice.

## 1 · Install

In [ ]:
%pip install -q "graphweave[llm,fast-knn] @ git+https://github.com/nevil-mathew/topic-extraction-poc.git@batch-clustering"
# datamapplot installed standalone so we don't pull the heavy graphweave[full] extra (torch/faiss/pacmap)
%pip install -q datamapplot
# static Matplotlib counterparts to the interactive charts below: a real Catppuccin .mplstyle + treemap layout
%pip install -q "catppuccin[matplotlib]" squarify
print("installs done — if Kaggle asks to restart the session, do it, then re-run from cell 2.")

## 2 · Config knobs (re-tune here)

These are the only values you normally touch. Embeddings are cached, so after the first run you can
re-tune `RESOLUTION` / `MIN_CLUSTER_FRACTION` and re-run from the **fit** cell without re-embedding.

In [ ]:
import os, glob, hashlib
import numpy as np
import pandas as pd

# --- data ---
# Point this at one or more uploaded Kaggle dataset copies of the CSV. Accepts a list, or a
# comma-separated string via the CSV_PATHS env var (e.g. multiple Kaggle dataset files).
_default_csv_paths = [
    "/kaggle/input/datasets/nevilmathew/csv-sample/challenges-a.csv","/kaggle/input/datasets/nevilmathew/csv-sample/challenges-b.csv"
]
_env_csv_paths = os.environ.get("CSV_PATHS") or os.environ.get("CSV_PATH")
if _env_csv_paths:
    CSV_PATHS = [p.strip() for p in _env_csv_paths.split(",") if p.strip()]
else:
    CSV_PATHS = _default_csv_paths

# Local fallback for running outside Kaggle: pick up every matching CSV in ~/Downloads.
if not any(os.path.exists(p) for p in CSV_PATHS):
    _local = sorted(glob.glob(os.path.expanduser("~/Downloads/report_list_*.csv")))
    if _local:
        CSV_PATHS = _local
WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# --- column names (config-driven; rename here if your CSV uses different headers) ---
# ID_COL and CHALLENGES_COL are REQUIRED. Everything else is OPTIONAL: if a column is
# missing from the CSV (or its config value is None), the notebook fills in a sensible
# default and keeps going — no District / date / etc. needed for a minimal run.
ID_COL         = "id"                   # unique row/report identifier
CHALLENGES_COL = "Challenges"            # free-text field, "|"-delimited, one or more challenges per row
DISTRICT_COL   = "District"              # optional grouping/overlay dimension; set to None to disable
DATE_COL       = "Date of Discussion"    # optional; set to None to disable

# --- embedding ---
EMBED_MODEL  = "BAAI/bge-base-en-v1.5"   # strong English encoder for short text (corpus is 99.5% English)
MULTILINGUAL = False                     # True -> swaps to BAAI/bge-m3 to also cluster the ~0.45% Hindi rows
EMBED_BATCH  = 64

# --- granularity (fine sub-topics; LLM merges them into parents afterwards) ---
RESOLUTION           = 1.5      # higher -> more, finer sub-topics
MIN_CLUSTER_FRACTION = 0.0005   # ~40-doc floor on 79k; suppresses noise specks

# --- accuracy vs runtime ---
N_CONSENSUS_RUNS = 10           # main accuracy lever; drop to ~5 for a fast preview
MAX_ITERATIONS   = 3            # iterative cluster<->embed refinement; 2 for a faster pass

# --- LLM (OpenRouter) for labels + topic merge ---
OPENROUTER_MODEL = "google/gemini-2.5-flash"   # cheap + capable; or "anthropic/claude-3.5-haiku"
DOMAIN_HINT      = "school & education challenges in rural Bihar, India"
N_PARENT_TOPICS  = None         # None = let the LLM choose natural parents (~15-30); or set an int

# --- LLM-guided granularity calibration (opt-in) ---
TUNE_RESOLUTION_WITH_LLM = False  # set True to auto-calibrate RESOLUTION via LLM
                                   # triplet judgments (ClusterLLM-style).
                                   # Uses OPENROUTER_MODEL — a cheap flash/DeepSeek
                                   # model is plenty for simple B-or-C judgments.

# --- result tables ---
N_EXAMPLES_PER_TOPIC = 20        # full-text example quotes shown per topic/sub-topic in the summary CSVs

# --- OpenRouter API key (Kaggle Secrets, then env var) ---
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    except Exception:
        OPENROUTER_API_KEY = None
USE_LLM = bool(OPENROUTER_API_KEY)

print("CSV_PATHS:")
for _p in CSV_PATHS:
    print(f"  {_p}  | exists: {os.path.exists(_p)}")
print("embed model:", "BAAI/bge-m3" if MULTILINGUAL else EMBED_MODEL)
print("LLM labeling/merge:", "ENABLED (OpenRouter)" if USE_LLM else "DISABLED (no key) -> keyword labels, single level")

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

# --- Catppuccin (Latte) palette: https://catppuccin.com/palette — used for every chart below,
# both the Plotly template and the matplotlib-side colour math in the datamapplot atlas cell. ---
CATPPUCCIN = {
    "base": "#eff1f5", "mantle": "#e6e9ef", "crust": "#dce0e8",
    "text": "#4c4f69", "subtext1": "#5c5f77", "subtext0": "#6c6f85",
    "overlay1": "#8c8fa1", "overlay0": "#9ca0b0",
    "surface2": "#acb0be", "surface1": "#bcc0cc", "surface0": "#ccd0da",
    "rosewater": "#dc8a78", "flamingo": "#dd7878", "pink": "#ea76cb",
    "mauve": "#8839ef", "red": "#d20f39", "maroon": "#e64553",
    "peach": "#fe640b", "yellow": "#df8e1d", "green": "#40a02b",
    "teal": "#179299", "sky": "#04a5e5", "sapphire": "#209fb5",
    "blue": "#1e66f5", "lavender": "#7287fd",
}
# categorical order: maximally distinct neighbours first, so the first ~8-10 series (the common case)
# stay easy to tell apart; cycles if a chart has more series than colours.
CATPPUCCIN_ACCENTS = [CATPPUCCIN[k] for k in (
    "mauve", "peach", "teal", "pink", "blue", "green",
    "yellow", "sapphire", "maroon", "lavender", "sky", "flamingo",
)]
# light-to-mauve continuous scale, for volume/heatmap-style colouring
CATPPUCCIN_SEQUENTIAL = [
    [0.0, CATPPUCCIN["base"]], [0.5, CATPPUCCIN["lavender"]], [1.0, CATPPUCCIN["mauve"]],
]

pio.templates["catppuccin"] = go.layout.Template(
    layout=go.Layout(
        colorway=CATPPUCCIN_ACCENTS,
        paper_bgcolor=CATPPUCCIN["base"],
        plot_bgcolor=CATPPUCCIN["base"],
        font=dict(family="Inter, Arial, sans-serif", color=CATPPUCCIN["text"], size=13),
        title=dict(font=dict(size=20, color=CATPPUCCIN["text"])),
        xaxis=dict(gridcolor=CATPPUCCIN["surface0"], linecolor=CATPPUCCIN["surface1"],
                    zerolinecolor=CATPPUCCIN["surface1"]),
        yaxis=dict(gridcolor=CATPPUCCIN["surface0"], linecolor=CATPPUCCIN["surface1"],
                    zerolinecolor=CATPPUCCIN["surface1"]),
        legend=dict(bgcolor=CATPPUCCIN["base"], bordercolor=CATPPUCCIN["surface0"], borderwidth=1),
        hoverlabel=dict(bgcolor=CATPPUCCIN["mantle"], font_color=CATPPUCCIN["text"],
                         bordercolor=CATPPUCCIN["surface1"]),
    )
)
pio.templates.default = "catppuccin"
print("registered 'catppuccin' Plotly template + palette (used by every chart below)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from catppuccin import PALETTE
from catppuccin.extras.matplotlib import get_colormap_from_list

# Static (PNG) counterpart to the Plotly "catppuccin" template above — same palette, applied via a
# real Catppuccin Latte .mplstyle (shipped by the catppuccin[matplotlib] package) rather than just
# setting rcParams inline. Used by every "*_static.png" cell below.
mpl.style.use(PALETTE.latte.identifier)
CATPPUCCIN_SEQ_CMAP = get_colormap_from_list(PALETTE.latte.identifier, ["base", "lavender", "mauve"])
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
print(f"matplotlib style: catppuccin {PALETTE.latte.identifier!r} — static PNGs saved next to the HTML files in WORKDIR")

## 3 · Load and explode challenges → one document per challenge

Each report's `Challenges` cell is split on `|`; every challenge becomes its own document. A parallel
`meta` frame (report id, District, date) stays aligned 1:1 with the documents to drive the District
overlay and hover text.

In [ ]:
import re

# only ID_COL and CHALLENGES_COL are required; everything else is read opportunistically
required_cols = [ID_COL, CHALLENGES_COL]
optional_cols = [c for c in (DISTRICT_COL, DATE_COL) if c]

if not CSV_PATHS:
    raise ValueError("CSV_PATHS is empty — set it in the config cell.")

# union of headers across all input CSVs: a column counts as "present" if at least one
# file has it; any file missing a REQUIRED column is reported and skipped, not silently dropped.
headers = {path: pd.read_csv(path, nrows=0, dtype=str).columns.tolist() for path in CSV_PATHS}
present_optional = sorted({c for h in headers.values() for c in h if c in optional_cols})
missing_optional = [c for c in optional_cols if c not in present_optional]
if missing_optional:
    print(f"optional column(s) not found in any input CSV, skipping: {missing_optional}")

usable_paths = []
for path, header in headers.items():
    missing_required = [c for c in required_cols if c not in header]
    if missing_required:
        print(f"skipping {path}: missing required column(s) {missing_required}")
        continue
    usable_paths.append(path)
if not usable_paths:
    raise ValueError("None of CSV_PATHS contain the required columns "
                      f"{required_cols}")

raw_frames = []
for path in usable_paths:
    present_here = [c for c in present_optional if c in headers[path]]
    frame = pd.read_csv(path, usecols=required_cols + present_here, dtype=str)
    for c in present_optional:
        if c not in frame.columns:
            frame[c] = pd.NA
    frame["source_file"] = os.path.basename(path)
    raw_frames.append(frame)
raw = pd.concat(raw_frames, ignore_index=True)

raw = raw.rename(columns={ID_COL: "report_id", CHALLENGES_COL: "Challenges"})
if DATE_COL and DATE_COL in present_optional:
    raw = raw.rename(columns={DATE_COL: "date"})
else:
    raw["date"] = pd.NA
if DISTRICT_COL and DISTRICT_COL in present_optional:
    raw = raw.rename(columns={DISTRICT_COL: "District"})
else:
    raw["District"] = "(unknown)"
raw["District"] = raw["District"].fillna("(unknown)").str.strip().replace("", "(unknown)")

# strip a leading list marker ("1.", "2)", "3 -", "4:") left over from numbered free-text fields
_LIST_PREFIX = re.compile(r"^\s*\d+\s*[.):-]\s*")

# explode pipe-delimited challenges, keeping metadata aligned
raw["Challenges"] = raw["Challenges"].fillna("").str.split("|")
ex = raw.explode("Challenges", ignore_index=True)
ex["Challenges"] = ex["Challenges"].str.strip().str.replace(_LIST_PREFIX, "", regex=True).str.strip()
ex = ex[ex["Challenges"].str.len() > 0].reset_index(drop=True)

documents = ex["Challenges"].tolist()
meta = ex[["report_id", "District", "date", "source_file"]].copy()

assert len(documents) == len(meta), "documents and meta must be 1:1"
print(f"input files: {len(usable_paths)}   reports: {raw['report_id'].nunique():,}   challenge documents: {len(documents):,}")
print("\nDocuments per source file:")
print(meta["source_file"].value_counts().to_string())
print("\nDistricts:")
print(meta["District"].value_counts().to_string())

## 4 · Embed once, cache to disk

Embedding is the expensive step. We compute it a single time and cache to `WORKDIR`, so re-tuning
clustering later does not re-embed. The cache key includes the model name and document count.

In [ ]:
from graphweave import GraphWeave, GraphWeaveConfig

embed_model = "BAAI/bge-m3" if MULTILINGUAL else EMBED_MODEL

cfg = GraphWeaveConfig(
    # --- embedding ---
    embedding_model=embed_model,
    embedding_batch_size=EMBED_BATCH,
    language="multilingual" if MULTILINGUAL else "english",
    # --- scale-critical (REQUIRED at ~79k; dense N×N consensus is infeasible) ---
    low_memory=True,
    consensus_method="graph",
    knn_backend="auto",          # -> hnswlib above 50k docs (needs the fast-knn extra)
    # --- graph (hybrid fuses semantic + SNN + lexical; lexical helps short text) ---
    graph_type="hybrid",
    use_lexical_view=True,
    lexical_weight=0.3,
    n_neighbors=20,
    # --- dim reduction for clustering ---
    use_dim_reduction=True,
    reduced_dims=10,
    umap_min_dist=0.0,
    # --- clustering (fine-grained) ---
    resolution=RESOLUTION,
    min_cluster_fraction=MIN_CLUSTER_FRACTION,
    n_consensus_runs=N_CONSENSUS_RUNS,
    # --- iterative refinement ---
    use_iterative_refinement=True,
    max_iterations=MAX_ITERATIONS,
    convergence_threshold=0.95,
    # --- keywords / misc ---
    keyword_method="ctfidf",
    n_keywords=10,
    random_state=42,
    verbose=True,
    n_jobs=-1,
)

model = GraphWeave(config=cfg)

cache_key = hashlib.md5(f"{embed_model}|{len(documents)}".encode()).hexdigest()[:10]
emb_path = os.path.join(WORKDIR, f"embeddings_{cache_key}.npy")
if os.path.exists(emb_path):
    embeddings = np.load(emb_path)
    print("loaded cached embeddings:", embeddings.shape)
else:
    embeddings = model.encode(documents)          # GPU sentence-transformers
    np.save(emb_path, embeddings)
    print("encoded + cached:", embeddings.shape, "->", emb_path)

## 5 · Full-batch fit (fine-grained)

One `fit()` over the complete consensus-Leiden graph. We pass the cached embeddings so this cell can
be re-run for granularity tuning without re-embedding.

In [ ]:
model.fit(documents, embeddings=embeddings)

labels = model.labels_
n_topics = len({l for l in labels if l != -1})
outlier_frac = float(np.mean(labels == -1))
stability = getattr(model, "stability_score_", None)

print(f"fine sub-topics: {n_topics}")
print(f"outliers: {outlier_frac:.1%}")
print(f"consensus stability: {stability if stability is None else round(stability, 3)} (target > ~0.8)")
print("\nLargest fine sub-topics:")
info = model.get_topic_info()
display(info[info.Topic != -1].sort_values("Size", ascending=False).head(15)[["Topic", "Size", "Keywords"]])

## 5b · Optional: LLM-guided granularity calibration

Instead of hand-tuning `RESOLUTION` in cell 2 and re-running, this asks the LLM
to judge a small number of boundary cases directly ("is document A more similar
to B or C?") and keeps whichever candidate resolution agrees with the LLM most
often — the **ClusterLLM** approach (Zhang, Wang & Shang, EMNLP 2023).

**Why it's better now:** the original port had a B-position bias (same-cluster
doc was always presented as B, inflating agreement with the middle candidate).
The improved version adds swap randomisation, discriminative anchor selection,
a two-stage grid that refines around the coarse winner with zero extra LLM calls,
and multi-seed averaging per candidate. The diagnostics table shows per-candidate
scores so a flat signal is visible at a glance.

Cost: ~$0.01–$0.15 with a flash/DeepSeek-class model. Opt-in: skipped unless
`TUNE_RESOLUTION_WITH_LLM = True` in cell 2 **and** an OpenRouter key is set.

In [ ]:
if USE_LLM and TUNE_RESOLUTION_WITH_LLM:
    from graphweave import LLMLabeler

    granularity_labeler = LLMLabeler(
        provider="openrouter",
        api_key=OPENROUTER_API_KEY,
        model=OPENROUTER_MODEL,   # cheap flash/DeepSeek-class model is plenty for B-or-C calls
        verbose=True,
    )
    model.tune_resolution_with_llm(
        granularity_labeler,
        resolution_range=(max(0.05, RESOLUTION * 0.4), RESOLUTION * 2.5),
    )

    labels = model.labels_
    n_topics = len({l for l in labels if l != -1})
    outlier_frac = float(np.mean(labels == -1))
    print(f"fine sub-topics after LLM calibration: {n_topics}")
    print(f"outliers: {outlier_frac:.1%}")
    print(f"calibrated resolution: {model.config.resolution:.4f}")

    # Diagnostics table — shows per-candidate scores so a flat signal is obvious
    if hasattr(model, 'granularity_diagnostics_'):
        diag = model.granularity_diagnostics_
        print(f"\ntriplets judged: {diag['n_triplets']}  "
              f"unparsed: {diag['n_unparsed']}  "
              f"reference res: {diag['reference_resolution']:.4f}")
        diag_df = pd.DataFrame(diag['candidates'])
        display(diag_df.sort_values('resolution').reset_index(drop=True)
                [['stage', 'resolution', 'score', 'n_clusters']])

    info = model.get_topic_info()
    display(info[info.Topic != -1].sort_values('Size', ascending=False)
            .head(15)[['Topic', 'Size', 'Keywords']])
else:
    print("skipped — set TUNE_RESOLUTION_WITH_LLM=True in cell 2 and provide "
          "OPENROUTER_API_KEY to enable")

**Re-tune if needed:** too few/huge topics → raise `RESOLUTION` (or lower `MIN_CLUSTER_FRACTION`);
too many tiny/noisy ones → the reverse. Edit cell 2 and re-run cells 5 onward (embeddings stay cached).

## 6 · Human-readable labels for the fine sub-topics

Friendly names (vs. raw keyword lists) are what make the shared map readable. Uses OpenRouter; falls
back to keyword-based `SimpleLabeler` if no API key was provided.

In [ ]:
from graphweave import LLMLabeler, SimpleLabeler

if USE_LLM:
    labeler = LLMLabeler(
        provider="openrouter",
        api_key=OPENROUTER_API_KEY,
        model=OPENROUTER_MODEL,
        domain_hint=DOMAIN_HINT,
        style="short",
        verbose=True,
    )
else:
    labeler = SimpleLabeler()

model.generate_labels(labeler)

# snapshot the FINE (sub-topic) labels before any merge
fine_label = {t.topic_id: (t.label or f"Topic {t.topic_id}")
              for t in model.topics_ if t.topic_id != -1}
fine_size  = {t.topic_id: t.size for t in model.topics_ if t.topic_id != -1}
print(f"labeled {len(fine_label)} sub-topics")
display(model.get_topic_info()[lambda d: d.Topic != -1]
        .sort_values("Size", ascending=False).head(15)[["Topic", "Size", "Label"]])

## 7 · LLM merge → parent topics (keeps both levels)

We call the low-level `llm_merge_topics_call` directly so it returns the merge **groups** without
mutating `model.labels_` — that way the fine sub-topics survive and we get a true two-level
**parent topic → sub-topic** structure. (`model.llm_merge_topics()` would collapse the fine level.)

In [ ]:
# fine_topic_id -> (parent_id, parent_label)
fine2parent_id, fine2parent_label = {}, {}

if USE_LLM and len(fine_label) > 1:
    from graphweave.labeling.llm_merger import llm_merge_topics_call

    topics_data = [
        {
            "topic_id": t.topic_id,
            "size": t.size,
            "keywords": t.keywords[: labeler.n_keywords],
            "representative_docs": [documents[i] for i in t.representative_docs[: labeler.n_docs]],
        }
        for t in model.topics_ if t.topic_id != -1
    ]
    groups = llm_merge_topics_call(
        labeler, topics_data, n_topics=N_PARENT_TOPICS,
        include_docs=True, use_structured_output=True,
    )
    for g in groups:
        ids = [int(i) for i in g["topic_ids"] if int(i) in fine_label]
        if not ids:
            continue
        parent_id = max(ids, key=lambda i: fine_size.get(i, 0))   # library convention: largest survives
        plabel = g.get("label") or fine_label[parent_id]
        for i in ids:
            fine2parent_id[i] = parent_id
            fine2parent_label[i] = plabel

# any fine topic the LLM didn't place (or LLM disabled) -> its own parent
for i, lab in fine_label.items():
    fine2parent_id.setdefault(i, i)
    fine2parent_label.setdefault(i, lab)

n_parents = len(set(fine2parent_id.values()))
print(f"{len(fine_label)} sub-topics  ->  {n_parents} parent topics\n")

# print the parent -> sub-topic tree
from collections import defaultdict
tree = defaultdict(list)
for fid, pid in fine2parent_id.items():
    tree[pid].append(fid)
for pid in sorted(tree, key=lambda p: -sum(fine_size[f] for f in tree[p])):
    psize = sum(fine_size[f] for f in tree[pid])
    print(f"■ {fine2parent_label[pid]}  ({psize:,} docs, {len(tree[pid])} sub-topics)")
    for fid in sorted(tree[pid], key=lambda f: -fine_size[f]):
        print(f"    └─ {fine_label[fid]}  ({fine_size[fid]:,})")

## 8 · Assemble result tables

In [ ]:
# keywords per fine topic, and the union of keywords across a parent's sub-topics
fine_keywords = {t.topic_id: t.keywords for t in model.topics_ if t.topic_id != -1}
parent_keywords = defaultdict(list)
for fid, pid in fine2parent_id.items():
    for kw in fine_keywords.get(fid, []):
        if kw not in parent_keywords[pid]:
            parent_keywords[pid].append(kw)

# per-document table
doc_df = meta.copy()
doc_df["text"]            = documents
doc_df["fine_topic"]      = labels
doc_df["fine_label"]      = [fine_label.get(l, "Outlier") for l in labels]
doc_df["fine_keywords"]   = [", ".join(fine_keywords.get(l, [])) for l in labels]
doc_df["parent_topic"]    = [fine2parent_id.get(l, -1) for l in labels]
doc_df["parent_label"]    = [fine2parent_label.get(l, "Unlabelled") for l in labels]
doc_df["parent_keywords"] = [", ".join(parent_keywords.get(fine2parent_id.get(l, -1), [])) for l in labels]
doc_df["is_outlier"]      = labels == -1

# per-parent-topic summary: size, top districts, example quotes, sub-topics
rows = []
clustered = doc_df[doc_df.fine_topic != -1]
for pid, grp in clustered.groupby("parent_topic"):
    top_districts = ", ".join(f"{d} ({c})" for d, c in grp.District.value_counts().head(3).items())
    subs = " | ".join(sorted({fine_label[f] for f in grp.fine_topic.unique()}))
    examples = " ⏐ ".join(grp.text.head(N_EXAMPLES_PER_TOPIC).tolist())
    rows.append({
        "Parent topic": fine2parent_label[pid],
        "Docs": len(grp),
        "Sub-topics": subs,
        "Top districts": top_districts,
        "Examples": examples,
    })
topic_summary = pd.DataFrame(rows).sort_values("Docs", ascending=False).reset_index(drop=True)

doc_df.to_csv(os.path.join(WORKDIR, "challenges_assignments.csv"), index=False)
topic_summary.to_csv(os.path.join(WORKDIR, "topic_summary.csv"), index=False)
print("saved challenges_assignments.csv and topic_summary.csv")
display(topic_summary)

## 9 · 2D projection for display (supervised by topic)

A separate UMAP tuned for *looking at* — and **supervised on the parent topic** so each topic lands as
one neat island instead of being scattered across the map (the big merged parents like *Aadhaar &
Documentation* otherwise spread out, since they're semantically diverse). Outliers pass in as `-1` and
are placed by content alone. `target_weight` (0→1) dials raw semantic layout vs. clean topic
separation; raise it toward 1.0 for tighter islands, lower it toward 0.0 for a more organic map. Both
interactive maps below reuse these coordinates.

In [ ]:
import umap

# Supervised UMAP: arrange the display *by parent topic* so each topic forms one neat island instead
# of being scattered across the map. We pass the parent-topic id as the target; outliers go in as -1,
# which UMAP treats as "unlabelled" (semi-supervised), so they're placed by content alone.
# target_weight trades off raw semantic structure (0.0) vs. clean topic separation (1.0).
y = doc_df["parent_topic"].to_numpy()

reducer = umap.UMAP(
    n_components=2, n_neighbors=20, min_dist=0.1, metric="cosine",
    target_metric="categorical", target_weight=0.5,
    random_state=42, verbose=True,
)
coords_2d = reducer.fit_transform(model.embeddings_, y=y)
doc_df["x"], doc_df["y"] = coords_2d[:, 0], coords_2d[:, 1]
print("display coords:", coords_2d.shape, "| supervised by parent topic (target_weight=0.5)")

## 10 · `datamapplot` atlas — the polished, shareable map ✨

A labeled **atlas** of the challenges, built for a "wow on open" reaction:

- **One colour per parent topic** — every point is coloured by its parent topic (sub-topics share the
  hue), so even a big, scattered parent like *Aadhaar & Documentation* reads as a single traceable
  colour you can spot anywhere on the map.
- **Two label layers** — zoomed out you read the ~parent topics; **zoom into any blob** and it names the
  fine sub-topic actually there. Labels sit on real points (medoids), not empty centroids.
- **Color-by selector** — recolour by **District** or **Sub-topic** from the dropdown.
- **Lasso select** — shift-drag any region to get a **word cloud** of those challenges.
- **Search + hover** — type a topic to highlight it; hover shows the District, sub-topic, and the real quote.

Saved as `challenges_atlas.html` — standalone HTML anyone can open.

In [ ]:
import colorsys
import matplotlib.colors as mcolors
import datamapplot
import datamapplot.selection_handlers

# --- two label layers: fine (zoom in) -> parent (zoom out) -------------------
# A big parent topic (e.g. "Aadhaar & Documentation", 17k) is a merge of several fine sub-topics that
# UMAP scatters across the map. Two fixes below make it findable anyway:
#   1) colour every point by its PARENT topic (one distinct colour per parent) so a scattered parent
#      reads as a single traceable colour — and with ~20 parents the colours are actually distinct,
#      unlike the default fine-layer colouring (100+ clusters -> near-identical pastels).
#   2) use_medoids=True places each label on a real data point, not the empty centroid between blobs.
fine_labels   = doc_df["fine_label"].replace("Outlier", "Unlabelled").to_numpy()
parent_labels = doc_df["parent_label"].to_numpy()
districts     = doc_df["District"].astype(str).to_numpy()

# --- per-sub-topic shade within each parent's hue ---------------------------
# Siblings under the same parent currently render as one flat colour (only the white grid
# line separates them in the treemap/Sankey). Vary saturation/value along the parent's own
# hue — not a new hue — so sub-topics are distinguishable but still read as the same family.
def _shade(hex_color, frac, sat_lo=0.35, sat_hi=0.75, val_lo=0.55, val_hi=0.92):
    r, g, b = mcolors.to_rgb(hex_color)
    h, _, _ = colorsys.rgb_to_hsv(r, g, b)
    s = sat_lo + frac * (sat_hi - sat_lo)
    v = val_hi - frac * (val_hi - val_lo)
    return mcolors.to_hex(colorsys.hsv_to_rgb(h, s, v))

# distinct Catppuccin accent per parent (largest parents first), shared by their sub-topics.
# Cycles through the 12 CATPPUCCIN_ACCENTS; once exhausted, reuses them via _shade() with a
# progressively different value so parents stay visually distinct past 12. (glasbey is always
# installed as a datamapplot dependency, so a try/except around it never actually fell back to
# the Catppuccin accents — go straight there instead.)
parent_order = (doc_df.loc[doc_df.parent_label != "Unlabelled", "parent_label"]
                .value_counts().index.tolist())
n_accents = len(CATPPUCCIN_ACCENTS)
parent_color = {}
for i, p in enumerate(parent_order):
    base = CATPPUCCIN_ACCENTS[i % n_accents]
    cycle = i // n_accents
    parent_color[p] = base if cycle == 0 else _shade(base, min(cycle / 3, 1.0))

sub_topic_color = {}
for pid, fids in tree.items():
    ordered = sorted(fids, key=lambda f: -fine_size[f])
    n = len(ordered)
    for idx, fid in enumerate(ordered):
        frac = idx / max(n - 1, 1)
        sub_topic_color[fine_label[fid]] = _shade(
            parent_color.get(fine2parent_label[fid], CATPPUCCIN["overlay1"]), frac)

marker_colors = np.array([parent_color.get(p, CATPPUCCIN["surface1"]) for p in parent_labels])

# label text colour: parent -> its colour; each sub-topic -> its parent's colour (so hues group)
label_color_map = dict(parent_color)
for _f, _p in zip(doc_df["fine_label"], doc_df["parent_label"]):
    label_color_map.setdefault(_f, parent_color.get(_p, CATPPUCCIN["overlay1"]))

# Plain-text hover (datamapplot escapes HTML): District · sub-topic, then the quote.
hover = (doc_df["District"].astype(str) + "  ·  " + doc_df["fine_label"].astype(str)
         + "\n" + doc_df["text"].astype(str).str.slice(0, 180)).to_numpy()

fig = datamapplot.create_interactive_plot(
    coords_2d,
    fine_labels,            # finest layer  -> shown when zoomed in
    parent_labels,          # coarsest layer -> shown when zoomed out
    hover_text=hover,
    noise_label="Unlabelled",
    title="School &amp; Education Challenges — Bihar",
    sub_title="Each point is one reported challenge, coloured by parent topic.",
    # colour every point by its parent topic; labels follow the same hues
    marker_color_array=marker_colors,
    label_color_map=label_color_map,
    use_medoids=True,                       # labels sit on a real point, not an empty centroid
    # alternate colourings available from the in-map selector
    colormaps={"District": districts, "Sub-topic": fine_labels},
    # shift-drag lasso -> word cloud of the selected challenges
    selection_handler=datamapplot.selection_handlers.WordCloud(
        256, width=400, height=320, n_rotations=0),
    # polish
    enable_search=True,
    darkmode=False,
    noise_color=CATPPUCCIN["surface1"],
    color_label_text=True,
    text_outline_width=1.5,
    font_family="Inter",
    extra_point_data=doc_df[["District", "fine_label", "parent_label", "report_id"]],
    inline_data=True,
)

atlas_path = os.path.join(WORKDIR, "challenges_atlas.html")
try:
    fig.save(atlas_path)
    print("saved", atlas_path)
except Exception as e:
    print("datamapplot atlas failed (", type(e).__name__, e, ") — the Plotly map below is the fallback.")

## 11 · Plotly map with a **District ↔ Topic** toggle

The analytical view: one dropdown flips the colouring between **parent topic** ("what are the themes")
and **District** ("how do districts differ"). Legend click isolates a series; hover shows the quote,
sub-topic and district. WebGL handles ~79k points smoothly. Saved as standalone HTML.

In [ ]:
import plotly.graph_objects as go

def _add_traces(frame, color_col, visible):
    traces, cats = [], frame[color_col].astype(str)
    for cat in sorted(cats.unique()):
        sub = frame[cats == cat]
        traces.append(go.Scattergl(
            x=sub.x, y=sub.y, mode="markers", name=str(cat)[:40], visible=visible,
            marker=dict(size=3, opacity=0.6),
            customdata=np.stack([sub.text.str.slice(0, 160), sub.fine_label, sub.District], axis=-1),
            hovertemplate="<b>%{customdata[2]}</b><br>%{customdata[1]}<br>%{customdata[0]}<extra></extra>",
        ))
    return traces

topic_traces = _add_traces(doc_df, "parent_label", True)
dist_traces  = _add_traces(doc_df, "District", False)
n_topic, n_dist = len(topic_traces), len(dist_traces)

fig = go.Figure(topic_traces + dist_traces)
fig.update_layout(
    title="Challenges map — colour by Topic or District",
    template="catppuccin", width=1100, height=750, legend=dict(itemsizing="constant"),
    updatemenus=[dict(
        type="dropdown", x=1.02, y=1.0, xanchor="left",
        buttons=[
            dict(label="Colour: Topic",    method="update",
                 args=[{"visible": [True]*n_topic + [False]*n_dist}]),
            dict(label="Colour: District", method="update",
                 args=[{"visible": [False]*n_topic + [True]*n_dist}]),
        ],
    )],
)
map_path = os.path.join(WORKDIR, "challenges_map.html")
fig.write_html(map_path, include_plotlyjs="cdn")
print("saved", map_path)
fig.show()

In [ ]:
# Static counterpart to the interactive dropdown above: two PNGs instead of one togglable figure.
def _static_scatter(color_col, color_map, path, title):
    fig, ax = plt.subplots(figsize=(11, 9))
    for cat, sub in doc_df.groupby(doc_df[color_col].astype(str)):
        ax.scatter(sub.x, sub.y, s=4, alpha=0.5, linewidths=0,
                   color=color_map.get(cat, CATPPUCCIN["overlay1"]), label=str(cat)[:30])
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(markerscale=4, fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    print("saved", path)


district_color = {d: CATPPUCCIN_ACCENTS[i % len(CATPPUCCIN_ACCENTS)]
                   for i, d in enumerate(doc_df.District.value_counts().index)}

_static_scatter("parent_label", parent_color,
                os.path.join(WORKDIR, "challenges_map_by_topic_static.png"),
                "Challenges map — coloured by Topic")
_static_scatter("District", district_color,
                os.path.join(WORKDIR, "challenges_map_by_district_static.png"),
                "Challenges map — coloured by District")

## 12 · Topic summary table (HTML for non-tech readers)

In [ ]:
table_html = topic_summary.to_html(index=False, escape=True)
styled = f'''<!doctype html><meta charset="utf-8">
<title>Challenge topics</title>
<style>body{{font-family:Inter,Arial,sans-serif;margin:24px;color:{CATPPUCCIN["text"]};
background:{CATPPUCCIN["base"]}}}
h1{{font-size:20px}} table{{border-collapse:collapse;width:100%;font-size:13px}}
th,td{{border:1px solid {CATPPUCCIN["surface1"]};padding:6px 8px;text-align:left;vertical-align:top}}
th{{background:{CATPPUCCIN["mantle"]}}} tr:nth-child(even){{background:{CATPPUCCIN["crust"]}}}</style>
<h1>School &amp; Education Challenges — topics &amp; sub-topics</h1>{table_html}'''
table_path = os.path.join(WORKDIR, "topic_summary.html")
with open(table_path, "w") as f:
    f.write(styled)
print("saved", table_path)

## 13 · Download & share

In Kaggle's right panel (**Output**), download these from `/kaggle/working`:

| File | What it is | For whom |
|---|---|---|
| `challenges_atlas.html` | Polished atlas — hull outlines, color-by Topic/District, lasso word-cloud, search | **Hand this to non-tech stakeholders** |
| `challenges_map.html` | Scatter with District ↔ Topic toggle | Analysts exploring district differences |
| `topic_summary.html` / `.csv` | Parent → sub-topics, sizes, top districts, examples | Plain-language reference |
| `challenges_assignments.csv` | Every challenge with its topic/sub-topic/district | Downstream analysis |
| `district_topic_heatmap.html` | District × Topic matrix, colour = volume, rich hover (share, sub-topics, example) | Fastest "which district has which problems" overview |
| `district_topic_treemap.html` | Click-to-drill District → Topic → Sub-topic, coloured by topic, sized by doc count | Analysts wanting exact counts + drill-down |
| `district_topic_sankey.html` | Topic → Sub-topic flow, pick a district from the dropdown | Deep dive into one district's topic mix |
| `district_topic_summary.html` / `.csv` | District → top topics/sub-topics, sizes, examples | Plain-language reference, per district |
| `topic_timeline.html` | Topic volume over time (stacked area, auto day/week/month buckets) | Spotting growing/shrinking problems |
| `topic_cooccurrence_network.html` | Network of parent topics that co-occur in the same report | Finding compound issues worth a joint fix |

All HTML files are self-contained — double-click to open in any browser, no install.

**Re-tuning:** change `RESOLUTION` / `MIN_CLUSTER_FRACTION` (granularity), set `MULTILINGUAL=True`
to cluster the Hindi rows, or `N_CONSENSUS_RUNS=5` / `MAX_ITERATIONS=2` for a faster preview — then
re-run from the **fit** cell (embeddings stay cached).

## 14 · District-level view: topics within each district

The maps above are organized **by topic** (each topic is one island; District is a secondary overlay
via the colour-by dropdown in `challenges_atlas.html`). This section flips the lens to answer
*"for each district, what are the topics/sub-topics, and how big are they"*:

- **`district_topic_heatmap.html`** — District × Topic matrix, colour = volume. The fastest way to
  compare many districts at once: position + colour intensity reads faster than nested-box area.
  Hover a cell for its share of the district, top sub-topics, and an example quote.
- **`district_topic_treemap.html`** — click-to-drill District → Topic → Sub-topic, sized by document
  count. Boxes are coloured **by topic** (same palette as the topic atlas), so a district's topic mix
  stays readable even when zoomed all the way in.
- **`district_topic_sankey.html`** — Topic → Sub-topic flow for **one district at a time**, picked from
  a dropdown. All districts at once (even capped to top-5/level) was still too dense to read, so this
  scopes to a single district per view — each one then shows its top ~8 topics / top ~6 sub-topics per
  topic with the rest bucketed into "Other", which stays comfortably readable.
- **`district_topic_summary.csv` / `.html`** — the District-first counterpart of section 8's
  `topic_summary`, for plain-language reference per district.

We dropped an earlier attempt at a District-supervised UMAP "atlas": District isn't a semantic/
embedding property the way topic is (two challenges from the same district aren't more alike in
meaning), so forcing it into spatial islands just scattered the map instead of revealing structure.

In [ ]:
# District x Topic heatmap: rows/cols sorted by total volume so the busiest districts/topics sit
# top-left. Colour = doc count. Hover carries share-of-district, top sub-topics, and a sample quote
# (substitutes for true click-to-filter, which would need custom JS callbacks in a static HTML export).
district_order_h = clustered.District.value_counts().index.tolist()
topic_order_h    = clustered.parent_label.value_counts().index.tolist()

pivot_count = (clustered.groupby(["District", "parent_label"]).size()
               .unstack(fill_value=0).reindex(index=district_order_h, columns=topic_order_h))
district_totals = pivot_count.sum(axis=1)
pivot_share = pivot_count.div(district_totals, axis=0).fillna(0)

# per-cell rich hover text: top sub-topics + one example quote
hover_cells = np.empty(pivot_count.shape, dtype=object)
for gi, district in enumerate(district_order_h):
    for gj, topic in enumerate(topic_order_h):
        cell = clustered[(clustered.District == district) & (clustered.parent_label == topic)]
        count = len(cell)
        if count == 0:
            hover_cells[gi, gj] = f"<b>{district}</b> · {topic}<br>0 challenges"
            continue
        subs = cell.fine_label.value_counts().head(3)
        sub_lines = "<br>".join(f"&nbsp;&nbsp;{s} ({c})" for s, c in subs.items())
        example = cell.text.iloc[0][:140]
        hover_cells[gi, gj] = (
            f"<b>{district}</b> · {topic}<br>{count:,} challenges "
            f"({pivot_share.iloc[gi, gj]:.1%} of district)<br>{sub_lines}<br><i>“{example}”</i>"
        )

fig_heat = go.Figure(go.Heatmap(
    z=pivot_count.values,
    x=topic_order_h,
    y=district_order_h,
    colorscale=CATPPUCCIN_SEQUENTIAL,
    hoverinfo="text",
    text=hover_cells,
    colorbar=dict(title="Docs"),
))
fig_heat.update_layout(
    title="Challenges by District × Topic (colour = volume)",
    template="catppuccin",
    xaxis=dict(tickangle=-45, title="Topic"),
    yaxis=dict(title="District", autorange="reversed"),
    width=1200, height=max(500, 28 * len(district_order_h)),
    margin=dict(t=60, l=140, r=40, b=160),
)

heatmap_path = os.path.join(WORKDIR, "district_topic_heatmap.html")
fig_heat.write_html(heatmap_path, include_plotlyjs="cdn")
print("saved", heatmap_path)
fig_heat.show()

In [ ]:
# Static counterpart to the interactive heatmap above; same pivot_count/order, no rich hover text.
fig_heat_s, ax = plt.subplots(figsize=(max(10, 0.5 * len(topic_order_h)), max(6, 0.3 * len(district_order_h))))
im = ax.imshow(pivot_count.values, cmap=CATPPUCCIN_SEQ_CMAP, aspect="auto")
ax.set_xticks(range(len(topic_order_h))); ax.set_xticklabels(topic_order_h, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(district_order_h))); ax.set_yticklabels(district_order_h, fontsize=8)
ax.set_title("Challenges by District × Topic (colour = volume)")
fig_heat_s.colorbar(im, ax=ax, label="Docs")
fig_heat_s.tight_layout()

heatmap_static_path = os.path.join(WORKDIR, "district_topic_heatmap_static.png")
fig_heat_s.savefig(heatmap_static_path)
plt.close(fig_heat_s)
print("saved", heatmap_static_path)

In [ ]:
# District -> Parent topic -> Sub-topic, sized by document count. Built with go.Treemap (not
# px.treemap) so each LEVEL gets its own colour rule: District boxes neutral grey, Topic boxes
# their flat parent colour, Sub-topic boxes a shade of that same colour (sub_topic_color, built
# in cell-21) — siblings become distinguishable without introducing a clashing new hue.
tree_df = (clustered.groupby(["District", "parent_label", "fine_label"])
           .size().reset_index(name="count"))

ids, labels_tm, parents_tm, values_tm, colors_tm = [], [], [], [], []

for district, dgrp in tree_df.groupby("District"):
    ids.append(district); labels_tm.append(district); parents_tm.append("")
    values_tm.append(int(dgrp["count"].sum())); colors_tm.append(CATPPUCCIN["surface0"])

    for parent, pgrp in dgrp.groupby("parent_label"):
        p_id = f"{district}/{parent}"
        ids.append(p_id); labels_tm.append(parent); parents_tm.append(district)
        values_tm.append(int(pgrp["count"].sum()))
        colors_tm.append(parent_color.get(parent, CATPPUCCIN["overlay1"]))

        for row in pgrp.itertuples():
            f_id = f"{district}/{parent}/{row.fine_label}"
            ids.append(f_id); labels_tm.append(row.fine_label); parents_tm.append(p_id)
            values_tm.append(int(row.count))
            colors_tm.append(sub_topic_color.get(row.fine_label, parent_color.get(parent, CATPPUCCIN["overlay1"])))

fig_treemap = go.Figure(go.Treemap(
    ids=ids, labels=labels_tm, parents=parents_tm, values=values_tm,
    branchvalues="total",
    marker=dict(colors=colors_tm, line=dict(width=2, color="white")),
    textfont=dict(size=14),
    pathbar=dict(textfont=dict(size=13)),
    hovertemplate="<b>%{label}</b><br>%{value:,} challenges<br>%{percentParent:.1%} of parent<extra></extra>",
))
fig_treemap.update_layout(
    title="Challenges by District → Topic → Sub-topic (click a box to drill in)",
    template="catppuccin",
    font=dict(size=13),
    title_font=dict(size=18),
    margin=dict(t=60, l=10, r=10, b=10),
    height=800,
)

treemap_path = os.path.join(WORKDIR, "district_topic_treemap.html")
fig_treemap.write_html(treemap_path, include_plotlyjs="cdn")
print("saved", treemap_path)
fig_treemap.show()

In [ ]:
# Static counterpart to the interactive Treemap above: same District -> Topic -> Sub-topic
# hierarchy and 3-tier colour rule (tree_df, parent_color, sub_topic_color all reused from above,
# no recomputation), laid out with squarify nested 3 levels deep (squarify itself only lays out one
# flat level, so each parent's rectangle is re-squarified for its children, and so on).
import squarify

FIG_W, FIG_H = 100.0, 60.0  # normalised layout units, independent of the actual figure size below


def _treemap_rects(sizes, x, y, dx, dy):
    if not sizes:
        return []
    return squarify.squarify(squarify.normalize_sizes(sizes, dx, dy), x, y, dx, dy)


fig_tm_s, ax = plt.subplots(figsize=(16, 10))

district_sizes = tree_df.groupby("District")["count"].sum().sort_values(ascending=False)
d_rects = _treemap_rects(district_sizes.values.tolist(), 0, 0, FIG_W, FIG_H)

for d_rect, district in zip(d_rects, district_sizes.index):
    ax.add_patch(plt.Rectangle((d_rect["x"], d_rect["y"]), d_rect["dx"], d_rect["dy"],
                                facecolor=CATPPUCCIN["surface0"], edgecolor="white", linewidth=2))
    if d_rect["dx"] > 4 and d_rect["dy"] > 4:
        ax.text(d_rect["x"] + 1, d_rect["y"] + d_rect["dy"] - 2, district,
                fontsize=9, fontweight="bold", va="top", color=CATPPUCCIN["text"])

    dgrp = tree_df[tree_df.District == district]
    p_sizes = dgrp.groupby("parent_label")["count"].sum().sort_values(ascending=False)
    p_rects = _treemap_rects(p_sizes.values.tolist(), d_rect["x"], d_rect["y"], d_rect["dx"], d_rect["dy"])

    for p_rect, parent in zip(p_rects, p_sizes.index):
        p_color = parent_color.get(parent, CATPPUCCIN["overlay1"])
        ax.add_patch(plt.Rectangle((p_rect["x"], p_rect["y"]), p_rect["dx"], p_rect["dy"],
                                    facecolor=p_color, edgecolor="white", linewidth=1.2))

        # sub-topic boxes: coloured (sub_topic_color) but left unlabelled — too many, too small to
        # read as text in a static image; the interactive Treemap above is the drill-down view.
        pgrp = dgrp[dgrp.parent_label == parent]
        f_sizes = pgrp.set_index("fine_label")["count"].sort_values(ascending=False)
        f_rects = _treemap_rects(f_sizes.values.tolist(), p_rect["x"], p_rect["y"], p_rect["dx"], p_rect["dy"])
        for f_rect, fine in zip(f_rects, f_sizes.index):
            ax.add_patch(plt.Rectangle((f_rect["x"], f_rect["y"]), f_rect["dx"], f_rect["dy"],
                                        facecolor=sub_topic_color.get(fine, p_color),
                                        edgecolor="white", linewidth=0.4))

        if p_rect["dx"] * p_rect["dy"] > 0.015 * FIG_W * FIG_H:
            ax.text(p_rect["x"] + 0.5, p_rect["y"] + p_rect["dy"] - 1, parent[:24],
                    fontsize=7, va="top", color=CATPPUCCIN["base"])

ax.set_xlim(0, FIG_W); ax.set_ylim(0, FIG_H)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Challenges by District → Topic → Sub-topic")
fig_tm_s.tight_layout()

treemap_static_path = os.path.join(WORKDIR, "district_topic_treemap_static.png")
fig_tm_s.savefig(treemap_static_path)
plt.close(fig_tm_s)
print("saved", treemap_static_path)

In [ ]:
TOP_TOPICS_PER_DISTRICT = 8   # per-district topic breadth shown before bucketing into "Other topics"
TOP_SUBS_PER_TOPIC      = 6   # per-topic sub-topic breadth before bucketing into "Other"

def _translucent(hex_color, alpha=0.5):
    """Plotly's Sankey link.color validator only accepts 6-digit hex / rgba() / named colors,
    not 8-digit hex — so blend alpha via an rgba() string instead of string-concatenating hex."""
    r, g, b = mcolors.to_rgb(hex_color)
    return f"rgba({int(r * 255)},{int(g * 255)},{int(b * 255)},{alpha})"

district_list = clustered.District.value_counts().index.tolist()

def _district_sankey_spec(district):
    """Build the node/link arrays for ONE district's Topic -> Sub-topic flow. Returned as a plain
    dict (not a go.Sankey trace) so the dropdown below can restyle a single shared trace in place —
    stacking one go.Sankey trace per district and toggling `visible` corrupts the link layout when
    you switch traces, since Plotly.js's Sankey auto-layout doesn't cleanly recompute across traces."""
    sub = clustered[clustered.District == district]
    topic_counts = sub.parent_label.value_counts()
    keep_topics = topic_counts.head(TOP_TOPICS_PER_DISTRICT).index.tolist()
    other_topic_count = int(topic_counts.iloc[TOP_TOPICS_PER_DISTRICT:].sum())

    node_labels, node_colors = [district], [CATPPUCCIN["text"]]

    def _add_node(label, color):
        node_labels.append(label)
        node_colors.append(color)
        return len(node_labels) - 1

    sources, targets, values, link_colors, link_hover = [], [], [], [], []

    for topic in keep_topics:
        t_idx = _add_node(topic, parent_color.get(topic, CATPPUCCIN["overlay1"]))
        t_count = int(topic_counts[topic])
        sources.append(0); targets.append(t_idx); values.append(t_count)
        link_colors.append(_translucent(parent_color.get(topic, CATPPUCCIN["overlay1"])))
        link_hover.append(f"{district} → {topic}: {t_count:,} challenges")

        tgrp = sub[sub.parent_label == topic]
        sub_counts = tgrp.fine_label.value_counts()
        keep_subs = sub_counts.head(TOP_SUBS_PER_TOPIC).index.tolist()
        other_sub_count = int(sub_counts.iloc[TOP_SUBS_PER_TOPIC:].sum())
        for s in keep_subs:
            sgrp = tgrp[tgrp.fine_label == s]
            # shade per sub-topic (sub_topic_color, built in cell-21) instead of a flat parent
            # colour, so siblings under the same topic node are distinguishable here too.
            s_idx = _add_node(s, sub_topic_color.get(s, parent_color.get(topic, CATPPUCCIN["overlay1"])))
            sources.append(t_idx); targets.append(s_idx); values.append(len(sgrp))
            link_colors.append(_translucent(parent_color.get(topic, CATPPUCCIN["overlay1"])))
            example = sgrp.text.iloc[0][:140]
            link_hover.append(f"{topic} → {s}: {len(sgrp):,} challenges<br><i>“{example}”</i>")
        if other_sub_count > 0:
            o_idx = _add_node("Other", parent_color.get(topic, CATPPUCCIN["overlay1"]))
            sources.append(t_idx); targets.append(o_idx); values.append(other_sub_count)
            link_colors.append(_translucent(parent_color.get(topic, CATPPUCCIN["overlay1"])))
            link_hover.append(f"{topic} → Other sub-topics: {other_sub_count:,} challenges")

    if other_topic_count > 0:
        o_idx = _add_node("Other topics", CATPPUCCIN["overlay1"])
        sources.append(0); targets.append(o_idx); values.append(other_topic_count)
        link_colors.append(_translucent(CATPPUCCIN["overlay1"]))
        link_hover.append(f"{district} → Other topics: {other_topic_count:,} challenges")

    return dict(
        node_label=node_labels, node_color=node_colors,
        link_source=sources, link_target=targets, link_value=values,
        link_color=link_colors, link_customdata=link_hover,
    )

district_specs = {d: _district_sankey_spec(d) for d in district_list}
first_district = district_list[0]
spec0 = district_specs[first_district]

# A SINGLE Sankey trace; the dropdown restyles its node/link arrays in place when you pick a
# district, instead of swapping which of many stacked traces is visible.
fig_sankey = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        label=spec0["node_label"], color=spec0["node_color"], pad=10, thickness=14,
        line=dict(width=0.5, color="white"),
        hovertemplate="%{label}<br>%{value:,} challenges<extra></extra>",
    ),
    link=dict(
        source=spec0["link_source"], target=spec0["link_target"], value=spec0["link_value"],
        color=spec0["link_color"], customdata=spec0["link_customdata"],
        hovertemplate="%{customdata}<extra></extra>",
    ),
))

buttons = []
for d in district_list:
    s = district_specs[d]
    buttons.append(dict(
        label=d, method="update",
        args=[
            {
                "node.label": [s["node_label"]],
                "node.color": [s["node_color"]],
                "link.source": [s["link_source"]],
                "link.target": [s["link_target"]],
                "link.value": [s["link_value"]],
                "link.color": [s["link_color"]],
                "link.customdata": [s["link_customdata"]],
            },
            {"title.text": f"Topic → Sub-topic flow — {d}"},
        ],
    ))

fig_sankey.update_layout(
    title=f"Topic → Sub-topic flow — {first_district}",
    template="catppuccin",
    font=dict(size=12),
    title_font=dict(size=18),
    width=1200, height=650,
    margin=dict(t=110, l=10, r=10, b=10),
    # top-LEFT, well clear of Plotly's own modebar (zoom/pan/save icons) which sits top-right.
    updatemenus=[dict(
        type="dropdown", x=0.0, y=1.2, xanchor="left", yanchor="top",
        buttons=buttons,
    )],
)

sankey_path = os.path.join(WORKDIR, "district_topic_sankey.html")
fig_sankey.write_html(sankey_path, include_plotlyjs="cdn")
print("saved", sankey_path, "|", len(district_list), "districts, pick one from the dropdown")
fig_sankey.show()

**Static Sankey: dropped.** Matplotlib has no native Sankey layout suited to a dropdown-driven,
per-district Topic → Sub-topic flow at this node count (`matplotlib.sankey` only draws simple linear
energy-balance diagrams, not multi-level flow trees) — see the interactive
`district_topic_sankey.html` above for this view.

In [ ]:
# District-first counterpart of topic_summary (cell-17): for each district, top topics, sub-topic
# breakdown, doc count, sample quotes.
rows = []
for district, grp in clustered.groupby("District"):
    top_topics = ", ".join(f"{t} ({c})" for t, c in grp.parent_label.value_counts().head(3).items())
    subs = " | ".join(sorted({fine_label[f] for f in grp.fine_topic.unique()})[:8])
    examples = " ⏐ ".join(grp.text.head(N_EXAMPLES_PER_TOPIC).tolist())
    rows.append({
        "District": district,
        "Docs": len(grp),
        "Top topics": top_topics,
        "Sub-topics": subs,
        "Examples": examples,
    })
district_topic_summary = pd.DataFrame(rows).sort_values("Docs", ascending=False).reset_index(drop=True)

district_topic_summary.to_csv(os.path.join(WORKDIR, "district_topic_summary.csv"), index=False)

table_html = district_topic_summary.to_html(index=False, escape=True)
styled = f'''<!doctype html><meta charset="utf-8">
<title>District topics</title>
<style>body{{font-family:Inter,Arial,sans-serif;margin:24px;color:{CATPPUCCIN["text"]};
background:{CATPPUCCIN["base"]}}}
h1{{font-size:20px}} table{{border-collapse:collapse;width:100%;font-size:13px}}
th,td{{border:1px solid {CATPPUCCIN["surface1"]};padding:6px 8px;text-align:left;vertical-align:top}}
th{{background:{CATPPUCCIN["mantle"]}}} tr:nth-child(even){{background:{CATPPUCCIN["crust"]}}}</style>
<h1>School &amp; Education Challenges — topics by District</h1>{table_html}'''
district_table_path = os.path.join(WORKDIR, "district_topic_summary.html")
with open(district_table_path, "w") as f:
    f.write(styled)

print("saved district_topic_summary.csv and district_topic_summary.html")
display(district_topic_summary)

## 15 · Topic volume over time

`DATE_COL` ("Date of Discussion") is carried through as `doc_df["date"]` but unused until now — this
answers *"is a problem growing or shrinking"*, which none of the maps above can show.

Dates are parsed defensively: `pd.to_datetime(..., errors="coerce")` turns anything unparseable
(blank cells, stray text, Excel serials read as strings, inconsistent formats) into `NaT` instead of
raising, and `dayfirst=True` matches the DD/MM/YYYY convention used in the source data. If too few
rows have a usable date, the cell prints why and skips the chart instead of producing a misleading
one. The bucket width (day/week/month) is chosen automatically from the date span so the chart stays
readable whether the data covers two weeks or two years.

In [ ]:
import warnings

TOP_TOPICS_TIMELINE = 10   # busiest parent topics shown individually; the rest bucket into "Other"

# Robust date parsing: errors="coerce" turns anything unparseable (blank cells, stray text,
# Excel-serial numbers read as strings, mixed formats) into NaT instead of raising. dayfirst=True
# matches the DD/MM/YYYY convention some source CSVs use; it's a no-op (with a harmless warning we
# suppress) on unambiguous ISO "%Y-%m-%d" dates, which is what this source's DATE_COL actually uses.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Parsing dates in.*when dayfirst=True was specified")
    parsed_dates = pd.to_datetime(clustered["date"], errors="coerce", dayfirst=True)
n_valid = int(parsed_dates.notna().sum())
n_total = len(parsed_dates)
print(f"valid dates: {n_valid:,}/{n_total:,} ({n_valid / max(n_total, 1):.1%})")

if n_valid < 10 or parsed_dates.dropna().nunique() < 2:
    print("too few valid/distinct dates to build a timeline — skipping "
          "(check DATE_COL in cell 2, or the source CSV's date formatting).")
else:
    tdf = clustered.loc[parsed_dates.notna()].copy()
    tdf["_date"] = parsed_dates.loc[parsed_dates.notna()]

    # pick a bucket width from the date span so the chart stays readable whether the data
    # covers two weeks (daily buckets) or several years (monthly buckets)
    span_days = (tdf["_date"].max() - tdf["_date"].min()).days
    freq = "D" if span_days <= 31 else "W" if span_days <= 365 else "MS"
    freq_label = {"D": "day", "W": "week", "MS": "month"}[freq]

    top_topics = tdf.parent_label.value_counts().head(TOP_TOPICS_TIMELINE).index.tolist()
    tdf["_topic_bucket"] = np.where(tdf.parent_label.isin(top_topics), tdf.parent_label, "Other")

    counts = (tdf.groupby([pd.Grouper(key="_date", freq=freq), "_topic_bucket"])
              .size().rename("count").reset_index())
    pivot = counts.pivot(index="_date", columns="_topic_bucket", values="count").fillna(0)
    col_order = pivot.sum().sort_values(ascending=False).index.tolist()   # busiest topic at the base
    pivot = pivot[col_order]

    fig_tl = go.Figure()
    for topic in col_order:
        color = parent_color.get(topic, CATPPUCCIN["overlay1"])
        fig_tl.add_trace(go.Scatter(
            x=pivot.index, y=pivot[topic], name=topic, mode="lines", stackgroup="one",
            line=dict(width=0.5, color=color), fillcolor=color,
            hovertemplate=f"<b>{topic}</b><br>%{{x|%Y-%m-%d}}: %{{y:.0f}} challenges<extra></extra>",
        ))

    fig_tl.update_layout(
        title=f"Topic volume over time (per {freq_label})",
        template="catppuccin",
        xaxis=dict(title="Date"), yaxis=dict(title="Challenges"),
        width=1200, height=600, legend=dict(itemsizing="constant"),
    )

    timeline_path = os.path.join(WORKDIR, "topic_timeline.html")
    fig_tl.write_html(timeline_path, include_plotlyjs="cdn")
    print("saved", timeline_path)
    fig_tl.show()

In [ ]:
# Static counterpart to the interactive timeline above; guarded the same way the interactive cell
# is (skips if there weren't enough valid/distinct dates to build pivot/col_order/freq_label).
if "pivot" not in globals():
    print("timeline was skipped above (too few valid dates) — skipping the static version too.")
else:
    fig_tl_s, ax = plt.subplots(figsize=(12, 6))
    ax.stackplot(pivot.index, [pivot[c] for c in col_order], labels=col_order,
                 colors=[parent_color.get(c, CATPPUCCIN["overlay1"]) for c in col_order])
    ax.set_title(f"Topic volume over time (per {freq_label})")
    ax.set_xlabel("Date"); ax.set_ylabel("Challenges")
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8, frameon=False)
    fig_tl_s.tight_layout()

    timeline_static_path = os.path.join(WORKDIR, "topic_timeline_static.png")
    fig_tl_s.savefig(timeline_static_path)
    plt.close(fig_tl_s)
    print("saved", timeline_static_path)

## 16 · Topic co-occurrence network

The Treemap and Sankey above are strictly **hierarchical** (topic → sub-topic); they can't show that
two *different* parent topics tend to show up **together in the same report** — e.g. a school that
reports an Aadhaar/documentation problem also reporting a mid-day-meal problem. This network answers
*"which problems travel together"*, which is useful for spotting compound issues worth a combined
intervention.

An edge connects two parent topics if they co-occur in the same `report_id` at least
`MIN_COOCCURRENCE` times; edge width = co-occurrence count, node size = topic volume, node colour
reuses the topic palette from the atlas. Built with `networkx` for the force-directed layout when
available, falling back to a simple circular layout (still correct, just less untangled) if it isn't
installed — so this cell never raises even on a minimal Kaggle image.</cell id="cell-33">


In [ ]:
from itertools import combinations
from collections import Counter

MIN_COOCCURRENCE = 3   # drop noise: topic pairs seen together in fewer than this many reports
MAX_EDGES         = 80  # cap for readability when there are many parent topics

# topics actually present in each report (excluding outliers); a set so a report mentioning the
# same topic via two different sub-topics still counts once per topic
report_topics = clustered.groupby("report_id")["parent_label"].apply(set)

pair_counts = Counter()
for topics in report_topics:
    if len(topics) < 2:
        continue
    for a, b in combinations(sorted(topics), 2):
        pair_counts[(a, b)] += 1

edges = [(a, b, c) for (a, b), c in pair_counts.items() if c >= MIN_COOCCURRENCE]
edges.sort(key=lambda e: -e[2])
edges = edges[:MAX_EDGES]

if not edges:
    print(f"no topic pairs co-occur >= {MIN_COOCCURRENCE} times within the same report — "
          "skipping the co-occurrence network (try lowering MIN_COOCCURRENCE).")
else:
    nodes = sorted({n for a, b, _ in edges for n in (a, b)})
    try:
        import networkx as nx
        G = nx.Graph()
        for a, b, c in edges:
            G.add_edge(a, b, weight=c)
        pos = nx.spring_layout(G, seed=42, k=1.5 / max(len(G), 1) ** 0.5, weight="weight")
    except ImportError:
        print("networkx not available — falling back to a circular layout (edges still correct).")
        pos = {n: (np.cos(2 * np.pi * i / len(nodes)), np.sin(2 * np.pi * i / len(nodes)))
               for i, n in enumerate(nodes)}

    topic_size = clustered.parent_label.value_counts()
    max_count = max(c for _, _, c in edges)
    max_size = topic_size.max()

    edge_traces = []
    for a, b, c in edges:
        x0, y0 = pos[a]; x1, y1 = pos[b]
        edge_traces.append(go.Scatter(
            x=[x0, x1], y=[y0, y1], mode="lines",
            line=dict(width=1 + 4 * c / max_count, color="rgba(140,143,161,0.4)"),  # catppuccin overlay1, translucent
            hoverinfo="text", text=f"{a} ↔ {b}: {c:,} reports", showlegend=False,
        ))

    node_list = sorted(nodes, key=lambda n: -topic_size.get(n, 0))
    node_trace = go.Scatter(
        x=[pos[n][0] for n in node_list], y=[pos[n][1] for n in node_list],
        mode="markers+text", text=node_list, textposition="top center",
        marker=dict(
            size=[8 + 22 * (topic_size.get(n, 1) / max_size) ** 0.5 for n in node_list],
            color=[parent_color.get(n, CATPPUCCIN["overlay1"]) for n in node_list],
            line=dict(width=1, color="white"),
        ),
        hovertext=[f"{n}: {topic_size.get(n, 0):,} challenges" for n in node_list],
        hoverinfo="text", showlegend=False,
    )

    fig_net = go.Figure(edge_traces + [node_trace])
    fig_net.update_layout(
        title="Topic co-occurrence — which problems show up together in the same report",
        template="catppuccin",
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        width=1100, height=800,
    )

    network_path = os.path.join(WORKDIR, "topic_cooccurrence_network.html")
    fig_net.write_html(network_path, include_plotlyjs="cdn")
    print("saved", network_path, "|", len(edges), "edges,", len(node_list),
          "topics, min co-occurrence:", MIN_COOCCURRENCE)
    fig_net.show()

In [ ]:
# Static counterpart to the interactive co-occurrence network above; guarded the same way the
# interactive cell is (skips if no topic pairs cleared MIN_COOCCURRENCE, so pos/edges/node_list/etc.
# were never bound).
if "pos" not in globals():
    print("no co-occurrence edges above threshold — skipping the static network too.")
else:
    fig_net_s, ax = plt.subplots(figsize=(11, 9))
    for a, b, c in edges:
        x0, y0 = pos[a]; x1, y1 = pos[b]
        ax.plot([x0, x1], [y0, y1], color=CATPPUCCIN["overlay1"], alpha=0.4,
                linewidth=1 + 4 * c / max_count, zorder=1)

    xs = [pos[n][0] for n in node_list]
    ys = [pos[n][1] for n in node_list]
    sizes = [(8 + 22 * (topic_size.get(n, 1) / max_size) ** 0.5) ** 2 for n in node_list]
    colors = [parent_color.get(n, CATPPUCCIN["overlay1"]) for n in node_list]
    ax.scatter(xs, ys, s=sizes, c=colors, edgecolors="white", linewidths=1, zorder=2)
    for n, x, y in zip(node_list, xs, ys):
        ax.annotate(n[:24], (x, y), fontsize=7, ha="center", va="bottom",
                    xytext=(0, 6), textcoords="offset points", color=CATPPUCCIN["text"])

    ax.set_title("Topic co-occurrence — which problems show up together in the same report")
    ax.axis("off")
    fig_net_s.tight_layout()

    network_static_path = os.path.join(WORKDIR, "topic_cooccurrence_network_static.png")
    fig_net_s.savefig(network_static_path)
    plt.close(fig_net_s)
    print("saved", network_static_path)